# Title

Panda tables probably would've been a good idea. Also... using a `linspace` instead of the range... which should be an easy fix

In [1]:
import glob
import re
import math
import random
import pprint as pp
import numpy as np
    
import k3d
from k3d import platonic
from k3d.colormaps import matplotlib_color_maps

from IPython.display import display, Markdown, Latex

def printmd(string):
    display(Markdown(string))
def ptex(string):
    display(Latex(string))

# for icosahedrons
PHI = 1.618033988749895

# a = [1,2,3]
# b = [*a, 4,5,6] # ... nice, except when it do special idiomatic thing

Requires running after completion

In [256]:
printmd(f'#### Heights\n\
\n\
| $H_{{min}}$ | $H_{{max}}$ | $H_{{turret}}$ | $H_{{hub}}$ |\n\
| ---: | ---: | ---: | ---: |\n\
| {H_MIN}m | {H_MAX}m | {H_TURRET}m | {H_HUB}m |')

printmd(f'#### Angle\n\
\n\
| $\\theta_{{min}}$ | $\\theta_{{max}}$ | \n\
| ---: | ---: |\n\
| ${TH_MIN_DEG}\\degree$ | ${TH_MAX_DEG}\\degree$ |$')

printmd(f'#### Potential Energy\n\
\n\
| Min |  | Max | Unit | Desc |\n\
| ---: | ---: | ---:| ---:| :---|\n\
| {(H_MIN*GP_M):.3f} | K | {(H_MAX*GP_M):.3f} | ${{kg}}\\frac{{m^2}}{{s^2}}$ | Ground | \n\
| {((H_MIN-H_TURRET)*GP_M):.3f} | K | {((H_MAX-H_TURRET)*GP_M):.3f} | ${{kg}}\\frac{{m^2}}{{s^2}}$ | Turret |\n\
| {(H_MIN-H_HUB):.3f} | K | {((H_MAX-H_HUB)*GP_M):.3f} | ${{kg}}\\frac{{m^2}}{{s^2}}$ | Hub |')

#### Heights

| $H_{min}$ | $H_{max}$ | $H_{turret}$ | $H_{hub}$ |
| ---: | ---: | ---: | ---: |
| 2.2m | 4.0m | 0.45m | 1.8288m |

#### Angle

| $\theta_{min}$ | $\theta_{max}$ | 
| ---: | ---: |
| $45.0\degree$ | $79.0\degree$ |$

#### Potential Energy

| Min |  | Max | Unit | Desc |
| ---: | ---: | ---:| ---:| :---|
| 4.620 | K | 8.400 | ${kg}\frac{m^2}{s^2}$ | Ground | 
| 3.675 | K | 7.455 | ${kg}\frac{m^2}{s^2}$ | Turret |
| 0.371 | K | 4.560 | ${kg}\frac{m^2}{s^2}$ | Hub |

In [257]:
printmd(f'#### Velocities\n\
\n\
| Min |  | Max | Unit | Desc |\n\
| ---: | :---: | ---:| ---:| :---|\n\
| {(vz_range_print[0]):.3f} | $\\mathit{{v_z}}$ | {(vz_range_print[1]):.3f}| $\\frac{{m}}{{s}}$                     | At turret plane      |\n\
| {(vz_range_hub_print[0]):.3f} | $\\mathit{{v_z}}$ | {(vz_range_hub_print[1]):.3f}| $\\frac{{m}}{{s}}$             | At hub plane           |\n\
| {(vz_avg_t2h_p[0]):.3f} | $\\mathit{{v_z}}$ | {(vz_avg_t2h_p[1]):.3f} |$\\frac{{m}}{{s}}$                        | Average $v_z$            |')

printmd(f'#### Time\n\
\n\
| Min |  | Max | Unit | Desc |\n\
| ---: | :---: | ---:| ---:| :---|\n\
| {(t_range_hub_p[0]):.3f} | $\\mathit{{t_{{hub,hub}}}}$ | {(t_range_hub_p[1]):.3f}| $s$                           | Hub plane to hub plane |\n\
| {(t_range_t2h_p[0]):.3f} | $\\mathit{{t_{{t2h}}}}$ | {(t_range_t2h_p[1]):.3f}| $s$                               | Turret to hub plane    |\n\
| {(t_range_hub_p[0] + t_range_t2h_p[0]):.3f} | $\\mathit{{t}}$ | {(t_range_hub_p[1] + t_range_t2h_p[1]):.3f}| $s$ | Total Time of Flight   |')

#### Velocities

| Min |  | Max | Unit | Desc |
| ---: | :---: | ---:| ---:| :---|
| 4.141 | $\mathit{v_z}$ | 5.898| $\frac{m}{s}$                     | At turret plane      |
| 1.907 | $\mathit{v_z}$ | 4.613| $\frac{m}{s}$             | At hub plane           |
| 3.024 | $\mathit{v_z}$ | 5.256 |$\frac{m}{s}$                        | Average $v_z$            |

#### Time

| Min |  | Max | Unit | Desc |
| ---: | :---: | ---:| ---:| :---|
| 0.778 | $\mathit{t_{hub,hub}}$ | 1.883| $s$                           | Hub plane to hub plane |
| 0.456 | $\mathit{t_{t2h}}$ | 0.262| $s$                               | Turret to hub plane    |
| 1.234 | $\mathit{t}$ | 2.145| $s$ | Total Time of Flight   |

In [258]:
printmd(f'#### Results\n\
\n\
| **Turret** | Min  |       | Max  |  **Hub** | Min  |       | Max  |\n\
|   ---:     | ---: | :---: | ---: |  ---: | ---: | :---: | ---: |\n\
| $\\theta_{{max}}$ | {(vx_th_max_p[0]):.3f}      | $\\mathit{{v_x}}$  |  {(vx_th_max_p[1]):.3f}    | $\\theta_{{max}}$   | {(vx_th_max_hub_p[0]):.3f} | $\\mathit{{v_x}}$ | {(vx_th_max_hub_p[1]):.3f} |\n\
|                   | {(vz_range_print[0]):.3f}   | $\\mathit{{v_z}}$  | {(vz_range_print[1]):.3f}  |                     | {(vz_range_hub_print[0]):.3f} | $\\mathit{{v_z}}$ | {(vz_range_hub_print[1]):.3f} |\n\
|                   | {(v_th_max_p[0]):.3f}       | $\\mathit{{v}}$    | {(v_th_max_p[1]):.3f}      |                     | {(v_th_max_hub_p[0]):.3f} | $\\mathit{{v}}$ | {(v_th_max_hub_p[1]):.3f} |\n\
| $\\theta_{{min}}$ | {(vx_th_min_p[0]):.3f}      | $\\mathit{{v_x}}$  |  {(vx_th_min_p[1]):.3f}    | $\\theta_{{min}}$   | {(vx_th_min_hub_p[0]):.3f} | $\\mathit{{v_x}}$ | {(vx_th_min_hub_p[1]):.3f} |\n\
|                   | {(vz_range_print[0]):.3f}   | $\\mathit{{v_z}}$  | {(vz_range_print[1]):.3f}  |                     | {(vz_range_hub_print[0]):.3f} | $\\mathit{{v_z}}$ | {(vz_range_hub_print[1]):.3f} |\n\
|                   | {(v_th_min_p[0]):.3f}       | $\\mathit{{v}}$    | {(v_th_min_p[1]):.3f}      |                     | {(v_th_min_hub_p[0]):.3f} | $\\mathit{{v}}$ | {(v_th_min_hub_p[1]):.3f} |\n\
\n\
#### Check Triangles\n\
\n\
| **Turret**        | $v_x^2$ | $v_z^2$ | $v^2$                                                  | **Hub** | $v_x^2$ | $v_z^2$ | $v^2$ |\n\
| ----:             | ----: | ----:   | ----:                                                    | ----:   | ----:   | ----:   | ----: |\n\
| $\\theta_{{max}}$ | {(vx_th_max_p[0]):.3f} | {(vz_range_print[0]):.3f} | {(v_th_max_p[0]):.3f} | $\\theta_{{min}}$ |  {(vx_th_max_hub_p[1]):.3f}  | {(vz_range_hub_print[1]):.3f} | {(v_th_max_hub_p[1]):.3f} |\n\
| $\\theta_{{min}}$ | {(vx_th_min_p[0]):.3f} | {(vz_range_print[0]):.3f} | {(v_th_min_p[0]):.3f} | $\\theta_{{min}}$ |  {(vx_th_min_hub_p[1]):.3f}  | {(vz_range_hub_print[1]):.3f} | {(v_th_min_hub_p[1]):.3f} |\n\
')

#### Results

| **Turret** | Min  |       | Max  |  **Hub** | Min  |       | Max  |
|   ---:     | ---: | :---: | ---: |  ---: | ---: | :---: | ---: |
| $\theta_{max}$ | 0.805      | $\mathit{v_x}$  |  1.147    | $\theta_{max}$   | 0.371 | $\mathit{v_x}$ | 0.897 |
|                   | 4.141   | $\mathit{v_z}$  | 5.898  |                     | 1.907 | $\mathit{v_z}$ | 4.613 |
|                   | 4.219       | $\mathit{v}$    | 6.009      |                     | 1.943 | $\mathit{v}$ | 4.699 |
| $\theta_{min}$ | 4.141      | $\mathit{v_x}$  |  5.898    | $\theta_{min}$   | 1.907 | $\mathit{v_x}$ | 4.613 |
|                   | 4.141   | $\mathit{v_z}$  | 5.898  |                     | 1.907 | $\mathit{v_z}$ | 4.613 |
|                   | 5.857       | $\mathit{v}$    | 8.341      |                     | 2.697 | $\mathit{v}$ | 6.523 |

#### Check Triangles

| **Turret**        | $v_x^2$ | $v_z^2$ | $v^2$                                                  | **Hub** | $v_x^2$ | $v_z^2$ | $v^2$ |
| ----:             | ----: | ----:   | ----:                                                    | ----:   | ----:   | ----:   | ----: |
| $\theta_{max}$ | 0.805 | 4.141 | 4.219 | $\theta_{min}$ |  0.897  | 4.613 | 4.699 |
| $\theta_{min}$ | 4.141 | 4.141 | 5.857 | $\theta_{min}$ |  4.613  | 4.613 | 6.523 |


## Constants

### Field

In [2]:
F_X = 325.611*2
F_Y = 158.844*2
F_Z = 180
FIELD_DIMS = [F_X/2., F_Y/2., F_Z, 1]
TO_BLUE = [-(F_X/2), -(F_Y/2), 0, 1]       # from origin (but flipped for translation)
TO_ORIGIN = [-TO_BLUE[0],-TO_BLUE[1],0,1]  # from blue

# both @ and * are doing the same thing...?
xf_blue = np.multiply(np.matrix([-1,1,-1,-1], dtype=np.float32).T @ np.matrix([1,1,1,1]), np.eye(4))
xf_blue[:,3] = np.matrix(TO_ORIGIN).T
xf_blue

matrix([[ -1.   ,  -0.   ,  -0.   , 325.611],
        [  0.   ,   1.   ,   0.   , 158.844],
        [ -0.   ,  -0.   ,  -1.   ,   0.   ],
        [ -0.   ,  -0.   ,  -0.   ,   1.   ]])

### Targets

Icosahedrons that should fit a ball in their interior

$(T_X,T_Y) = (182.11,158.844)$

In [3]:
T_X = 182.11
T_Y = 158.844
T_Z = 72
TARGET_SIZE = 5.91; # 5.91/PHI;
TARGET_BLUE = [T_X, T_Y, T_Z, 1]
target_blue = platonic.Icosahedron(origin=TARGET_BLUE[0:3], size=(5.91/PHI))
target_blue.mesh.color = 0xFF0000
TARGET_RED = [(F_X-T_X), (F_Y-T_Y), T_Z, 1]
target_red = platonic.Icosahedron(origin=TARGET_RED[0:3], size=(5.91/PHI))
target_red.mesh.color = 0x0000FF

# Trajectory Calculations

started with reasonable heights. `> 4.0m` too high. arcs take more time. `45cm` expected turret height offsets potential energy.  Calculated $V_{z range}$ offset from turret & hub

In [4]:
# for a single-dim space anyways

def linspace_minmax(nparr):
    lsp_length=h_range.shape[1]
    return [nparr[0,0].item(), nparr[0,lsp_length-1].item()]

# linspace_minmax(k_range)

Game Piece

In [5]:
# Game Piece Radius, Mass
GP_R, GP_M = 5.91/2, 2.10 # to 2.27

In [ ]:
M_TO_IN = (100/2.54)
H_MIN = 2.2
H_MAX = 4.0
h_range = np.matrix([H_MIN,H_MAX])
h_range_in = h_range * M_TO_IN

In [54]:
G = -4.9
H_TURRET = 0.45
H_HUB = 72/M_TO_IN

ptex(f'$(H_{{min}}, H_{{max}}) = ({H_MIN}m, {H_MAX}m)$')
ptex(f'$(H_{{turret}},H_{{hub}}) = ({H_TURRET}m, {H_HUB}m)$')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [69]:
TH_MIN,TH_MAX = 45*(math.pi/180),79*(math.pi/180)
TH_MIN_DEG,TH_MAX_DEG = (TH_MIN*180/math.pi),(TH_MAX*180/math.pi)

ptex(f'$(\\theta_{{min}}, \\theta_{{max}}) = {TH_MIN_DEG}\\degree,{TH_MAX_DEG}\\degree$')

<IPython.core.display.Latex object>

In [ ]:
k_range = GP_M * (h_range - H_TURRET) # kg * m
k_range_hub = GP_M * (h_range - H_HUB) # kg * m

## Potential Energy

In [103]:
ptex(f'At peak, w.r.t')
ptex(f'$Ground: ({(H_MIN*GP_M):.3f} < K < {(H_MAX*GP_M):.3f}) ~{{kg}}\\cdot{{m}}$') # < {k_range[1]} kg*m')
ptex(f'$Turret: ({((H_MIN-H_TURRET)*GP_M):.3f} < K < {((H_MAX-H_TURRET)*GP_M):.3f}) ~{{kg}}\\cdot{{m}}$') # < {k_range[1]} kg*m')
ptex(f'$Hub: ({(H_MIN-H_HUB):.3f} < K < {((H_MAX-H_HUB)*GP_M):.3f}) ~{{kg}}\\cdot{{m}}$') # < {k_range[1]} kg*m')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## $V_z$ Range

In [131]:
vz_range = np.sqrt(abs(2*G*(h_range - H_TURRET))) # from turret
vz_range_hub = np.sqrt(abs(2*G*(h_range - H_HUB))) # from turret

vz_range_print = linspace_minmax(vz_range)
vz_range_hub_print = linspace_minmax(vz_range_hub)

ptex(f'$At~turret~plane: ({(vz_range_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_print[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$At~hub~plane: ({(vz_range_hub_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_hub_print[1]):.3f}) ~\\frac{{m}}{{s}}$')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

## Flight Times

From peak to hub plane, then double to get "hub to hub" time

In [183]:
t_range_hub_half = np.sqrt(2*(h_range - H_HUB)/-G)
t_range_hub = 2 * t_range_hub_half

t_range_hub_half_p = linspace_minmax(t_range_hub_half)
t_range_hub_p = linspace_minmax(t_range_hub)

ptex(f'$Peak~to~hub~plane: ({(t_range_hub_half_p[0]):.3f} < \\mathit{{t_{{hub,hub}} / 2}} < {(t_range_hub_half_p[1]):.3f}) ~s$')
ptex(f'$Hub~plane~to~hub~plane: ({(t_range_hub_p[0]):.3f} < \\mathit{{t_{{hub,hub}}}} < {(t_range_hub_p[1]):.3f}) ~s$')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

Calculate average $v_z$ to work back to $t_{t2h}$, the time from launch to first crossing the hub plane.

In [175]:
vz_avg_t2h = (vz_range_hub + vz_range)/2
t_range_t2h = [H_HUB-H_TURRET,H_HUB-H_TURRET] / vz_avg_t2h

vz_avg_t2h_p = linspace_minmax(vz_avg_t2h)
t_range_t2h_p = linspace_minmax(t_range_t2h)

ptex(f'$Average~v_z: ({(vz_avg_t2h_p[0]):.3f} < \\mathit{{v_z}}~ < {(vz_avg_t2h_p[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$Turret~to~hub~plane: ({(t_range_t2h_p[0]):.3f} < \\mathit{{t_{{t2h}}}} < {(t_range_t2h_p[1]):.3f}) ~s$')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [176]:
ptex(f'$Total~Time~of~Flight: ({(t_range_hub_p[0] + t_range_t2h_p[0]):.3f} < \\mathit{{t}} < {(t_range_hub_p[1] + t_range_t2h_p[1]):.3f}) ~s$')

<IPython.core.display.Latex object>

Calculating time to hub height gets a symmetric parabola.

## Law of Sines

$$\left(\frac{|v|}{sin(\frac{\pi}{2})}\right) = \left(\frac{v_x}{sin(\theta)}\right) = \left(\frac{v_z}{sin(\frac{\pi}{2} - \theta)}\right)~,~ \theta < \frac{\pi}{2}$$ 

<p style="visibility: hidden;">
it was used around here somewhere. i'm sure of it... ahh there it is</p>

$$\frac{v_z}{|v|} = \sin\theta~~\rightarrow~~\frac{v_z \sin(\frac{\pi}{2} - \theta)}{\sin\theta}$$

$$\frac{v_z \cos\theta}{\sin\theta} = v_z\cot(\theta) = v_x$$

In [177]:
vx_th_max = (1/math.tan(TH_MAX))*vz_range
v_th_max = vz_range/math.sin(TH_MAX)
vx_th_min = (1/math.tan(TH_MIN))*vz_range
v_th_min = vz_range/math.sin(TH_MIN)

# and print
vx_th_max_p = linspace_minmax(vx_th_max)
v_th_max_p = linspace_minmax(v_th_max)
vx_th_min_p = linspace_minmax(vx_th_min)
v_th_min_p = linspace_minmax(v_th_min)

# and now actually print

ptex('$At~\\theta_{{max}}$')
ptex(f'$v_x: ({(vx_th_max_p[0]):.3f} < \\mathit{{v_x}}~ < {(vx_th_max_p[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v_z: ({(vz_range_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_print[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v: ({(v_th_max_p[0]):.3f} < \\mathit{{v}}~ < {(v_th_max_p[1]):.3f}) ~\\frac{{m}}{{s}}$')

ptex('$At~\\theta_{{min}}$')
ptex(f'$v_x: ({(vx_th_min_p[0]):.3f} < \\mathit{{v_x}}~ < {(vx_th_min_p[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v_z: ({(vz_range_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_print[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v: ({(v_th_min_p[0]):.3f} < \\mathit{{v}}~ < {(v_th_min_p[1]):.3f}) ~\\frac{{m}}{{s}}$')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [160]:
vx_th_max_hub = (1/math.tan(TH_MAX))*vz_range_hub
v_th_max_hub = vz_range_hub/math.sin(TH_MAX)
vx_th_min_hub = (1/math.tan(TH_MIN))*vz_range_hub
v_th_min_hub = vz_range_hub/math.sin(TH_MIN)

vx_th_max_hub_p = linspace_minmax(vx_th_max_hub)
v_th_max_hub_p = linspace_minmax(v_th_max_hub)
vx_th_min_hub_p = linspace_minmax(vx_th_min_hub)
v_th_min_hub_p = linspace_minmax(v_th_min_hub)

# and now actually print
ptex('$At~\\theta_{{max}}$')
ptex(f'$v_x: ({(vx_th_max_hub_p[0]):.3f} < \\mathit{{v_x}}~ < {(vx_th_max_hub_p[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v_z: ({(vz_range_hub_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_hub_print[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v: ({(v_th_max_hub_p[0]):.3f} < \\mathit{{v}}~ < {(v_th_max_hub_p[1]):.3f}) ~\\frac{{m}}{{s}}$')

ptex('$At~\\theta_{{min}}$')
ptex(f'$v_x: ({(vx_th_min_p[0]):.3f} < \\mathit{{v_x}}~ < {(vx_th_min_p[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v_z: ({(vz_range_hub_print[0]):.3f} < \\mathit{{v_z}} < {(vz_range_hub_print[1]):.3f}) ~\\frac{{m}}{{s}}$')
ptex(f'$v: ({(v_th_min_p[0]):.3f} < \\mathit{{v}}~ < {(v_th_min_p[1]):.3f}) ~\\frac{{m}}{{s}}$')


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### Check the triangles

In [161]:
["v min/max @ theta max", 
 np.multiply(vx_th_max,vx_th_max), 
 np.multiply(vz_range,vz_range),
 np.multiply(v_th_max,v_th_max),
 "check",
 np.multiply(v_th_max,v_th_max) - np.multiply(vx_th_max,vx_th_max),
 "v min/max @ theta min",
 np.multiply(vx_th_min,vx_th_min), 
 np.multiply(vz_range,vz_range),
 np.multiply(v_th_min,v_th_min),
 "check",
 np.multiply(v_th_min,v_th_min) - np.multiply(vx_th_min,vx_th_min)]

['v min/max @ theta max',
 matrix([[0.64799053, 1.31449508]]),
 matrix([[17.15, 34.79]]),
 matrix([[17.79799053, 36.10449508]]),
 'check',
 matrix([[17.15, 34.79]]),
 'v min/max @ theta min',
 matrix([[17.15, 34.79]]),
 matrix([[17.15, 34.79]]),
 matrix([[34.3 , 69.58]]),
 'check',
 matrix([[17.15, 34.79]])]

# Plot

### Robot pose

In [349]:
robot_pose = np.matrix([1.0, 1.0, H_TURRET]) * M_TO_IN
robot_pose_delta = TARGET_BLUE[0:3] - robot_pose
robot_pose_theta = math.atan(robot_pose_delta[0,1]/robot_pose_delta[0,0])
# robot_pose_theta * 180/math.pi # 39.929 degrees

sin_th,cos_th = math.sin(robot_pose_theta),math.cos(robot_pose_theta)

xlate = np.transpose(robot_pose)
# xlate = robot_pose
xrot = np.matrix([
    [1, 0, 0],
    [0, cos_th, -sin_th],
    [0, sin_th, cos_th]
])

# nvm: this rotates, then translates...
# xform = np.vstack([np.hstack([xrot, xlate]), [0,0,0,1]])


matrix([[39.37007874],
        [39.37007874],
        [17.71653543]])

### Generate Data

In [ ]:
total_t = t_range_hub + t_range_t2h

graph_t = np.linspace(0, total_t, 100, dtype=np.float32)
graph_z = (G/2 * np.power(graph_t,2)) + np.multiply(vz_range,graph_t) + H_TURRET
graph_y = np.zeros(graph_t.shape)
graph_x_th_max = np.multiply(vx_th_max,graph_t)
graph_x_th_min = np.multiply(vx_th_min,graph_t) + H_TURRET

graph_z = graph_z * M_TO_IN
graph_x_th_max = graph_x_th_max * M_TO_IN
graph_x_th_min = graph_x_th_min * M_TO_IN

t1 = np.hstack([graph_x_th_max, graph_y, graph_z])
t2 = np.hstack([graph_x_th_min, graph_y, graph_z])
t2

xrot_ohno = np.dstack([xrot.diagonal(),xrot.diagonal()]).reshape(3,2).T
xrot_ohno

# np.multiply()

# robot_pose_theta = 
# np.hstack([graph_t, graph_z, graph_y, graph_x_th_max, graph_x_th_max])
# robot_pose_theta

# h_range_in

# t = np.linspace(t_, 10, 100,dtype=np.float32)
# x = np.cos(t)
# y = np.sin(t)
# z = t / 5

# vertices = np.vstack([x,y,z]).T

# plt_line = k3d.line(vertices, width=0.1, shader='mesh',
#                 color_map=matplotlib_color_maps.Jet,
#                 attribute=t,
#                 color_range=[-5, 5])

# STL

In [11]:
def stl_props(f, name=None, group=None, **kwargs):
    rxgroup = r'stl/([^_]+)_.*.stl'
    rxname = r'stl/[^_]+_(.*).stl'
    name = (name or re.sub(rxname, r'\1', f))
    group = (group or re.sub(rxgroup, r'\1', f))
    return dict(name=name, group=group, **kwargs)
np.column_stack
# flat_shading = True, shininess = 50.0, compression_level = 0
def stl_read(f:str, color = 0x0000ff, wireframe = False, name = None, group = None, custom_data = None):
    with open(f, 'r') as stl:
        data = stl.read()
    return k3d.stl(data, color=color, wireframe=wireframe, name=name, group=group, custom_data=custom_data)

In [12]:
stl_blue = [stl_read(f, **stl_props(f, color=0x000066)) for f in glob.glob('stl/blue_*.stl') ]
stl_red = [stl_read(f, **stl_props(f), color=0x660000) for f in glob.glob('stl/red_*.stl') ]
stl_neutral = [stl_read(f, **stl_props(f, color=0xaaaaaa)) for f in glob.glob('stl/neutral_*.stl') ]
stl_all = (stl_blue + stl_red + stl_neutral)

for s in stl_all: s.model_matrix = xf_blue

/usr/local/lib/python3.12/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(


In [ ]:
plot = k3d.plot()
plot += plt_line
plot.display()

In [13]:



plot = k3d.plot()
# for s in (stl_blue + stl_red + stl_neutral): plot += s
for s in stl_all: plot += s
plot += target_blue.mesh
plot += target_red.mesh

plot.display()

Output()

## Misc

In [14]:
def randomblobs(n=10):
    blobs = []
    for i in range(n): 
        rndpoint = [F_X*random.gauss(0.5,0.25),
                    F_Y*random.gauss(0.5,0.25), 
                    F_Z*random.gauss(0.75,0.125)]
        b = platonic.Icosahedron(origin=rndpoint, size=(5/PHI))
        
        b.mesh.color = 0xFFFF00
        blobs.append(b)
    return blobs

### Updating live plots

- `display()` re-renders the plot in a new cell
- `render()` doesn't work across cells unless the plot is in "callback" mode
- better info in [time series](https://k3d-jupyter.org/user/time.html)
- can pre-generate time-series data for animation (like in [hydrogen_orbitals.ipynb](https://github.com/K3D-tools/K3D-jupyter/blob/main/examples/hydrogen_orbitals.ipynb))

In [15]:
# for b in randomblobs(100): plot += b.mesh
# plot.render() 

In [16]:
bb = randomblobs(5)
type(bb[0].mesh)

# bb[0].mesh.set_trait(color=0xFFFF00)

k3d.objects.geometry.Mesh

In [17]:
# rndpoint = [F_X*random.gauss(0.5,0.25),
#                     F_Y*random.gauss(0.5,0.25), 
#                     F_Z*random.gauss(0.75,0.125)]
# rndpoint

In [18]:
# stl_blue_zone = stl_read('frc_2026_field_zones_3d.stl', **stl_props('frc_2026_field_zones_3d.stl', color=0x666600))
# stl_blue_zone = stl_read('stl/blue_tower.stl', **stl_props('stl/blue_tower.stl', color=0x666600))
stl_blue_zone = stl_read('stl/blue_zone.stl', **stl_props('stl/blue_zone.stl', color=0x666600))

plot2 = k3d.plot()
# for s in (stl_blue + stl_red + stl_neutral): plot += s
plot2 += stl_blue_zone
plot2.display()

Output()